**Colab version of `CTD Python Data Pipeline (capstone)`.** Same code as the Kaggle notebook; the first cell downloads the Kaggle datasets with `kagglehub` (no Kaggle login needed for public datasets). Runtime → Run all to reproduce.

Submission hub with every link for this week: https://github.com/ba-00001/brian-bazurto-python-v2/blob/assignment6/final-project/README.md

In [ ]:
# Google Colab setup: Kaggle "Add Input" folders don't exist in Colab, so this cell
# downloads the same public Kaggle datasets with kagglehub and copies them into
# /content/kaggle_input (Colab's /kaggle/input is read-only). The rest of the
# notebook reads from KAGGLE_INPUT, so it runs the same as on Kaggle.
import os
import shutil

import numpy as np
import pandas as pd

try:
    import kagglehub
except ImportError:
    %pip install -q kagglehub
    import kagglehub

DATASETS = ['anandaramg/global-superstore']

KAGGLE_INPUT = '/content/kaggle_input'
os.makedirs(KAGGLE_INPUT, exist_ok=True)
for slug in DATASETS:
    downloaded = kagglehub.dataset_download(slug)
    target = os.path.join(KAGGLE_INPUT, slug.split('/')[1])
    shutil.copytree(downloaded, target, dirs_exist_ok=True)

for dirname, _, filenames in os.walk(KAGGLE_INPUT):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# CTD Python Data Pipeline: Global Superstore

**Author:** Brian Bazurto (GitHub: ba-00001)
**Course:** Code the Dream, Python 100 (final project / capstone)

## Question and goal

The Global Superstore dataset has 51,290 order line items from a large retailer that sells furniture, office supplies and technology in 147 countries (2011 to 2014). I study business analytics and supply chain, so I want to use it to answer one main question:

> **What actually drives profit, as opposed to just sales?**

The sub-questions I am working toward:

1. **Discounts:** how much do discounts hurt profit margin, and is there a point where they turn orders into losses?
2. **Products:** which categories and sub-categories make money and which lose it?
3. **Geography:** which markets and regions are the most and least profitable?
4. **Shipping:** how do ship mode and order priority relate to shipping time and cost?

## Dataset

- **Source:** [Global Superstore on Kaggle](https://www.kaggle.com/datasets/anandaramg/global-superstore/data) (uploaded by anandaramg), file `Global Superstore.txt`.
- **Format:** tab-separated text, UTF-8, 51,290 rows x 27 columns.
- **Grain:** one row per product line within an order (`Row ID`). An `Order ID` can have several rows.
- **Columns:** order and ship dates, ship mode, order priority, customer and segment, location (city / state / country / market / region), product and category, and the numeric fields `Sales`, `Quantity`, `Discount`, `Profit`, `Shipping Cost`. It also has a few helper columns (`Year`, `weeknum`, `Market2` and a record-count column) that I look at in the cleaning section.

## How this notebook is built

This project is built incrementally, one section per week:

| Week | Section | Status |
|---|---|---|
| 5 | Loading, inspection and data cleaning | Done |
| 6 | Data wrangling, aggregation and feature engineering | Done |
| 7+ | Visualization, analysis and conclusions | Coming later |

## Setup

Imports and display settings. Everything used here comes preinstalled on Kaggle (pandas, numpy, matplotlib).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # used for the plots in Week 7+

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

print("pandas", pd.__version__, "| numpy", np.__version__)

---
# Week 5: Loading, Inspection and Cleaning

## 1. Loading the data

The Kaggle input path depends on how the dataset was added (on my run it was `/kaggle/input/datasets/anandaramg/global-superstore/Global Superstore.txt`), so I do not hard-code it. The function below walks `/kaggle/input` and picks up any file with "superstore" in the name and a `.txt`, `.csv` or `.xlsx` extension. If the notebook runs outside Kaggle it falls back to a local `./data/` folder.

The Kaggle file is a **tab-separated** `.txt`, so it needs `sep="\t"`. With the default comma separator everything would end up in one column. The loader chooses the separator from the extension and also handles a comma CSV or an Excel copy of the same data.

In [ ]:
SEARCH_DIRS = [KAGGLE_INPUT, "./data", "."]
EXT_PRIORITY = [".txt", ".csv", ".xlsx"]  # prefer the original Kaggle .txt


def find_data_file(search_dirs=SEARCH_DIRS, keyword="superstore"):
    """Return the first data file whose name contains `keyword` (searched recursively)."""
    for base in search_dirs:
        if not os.path.isdir(base):
            continue
        matches = []
        for root, _, files in os.walk(base):
            for f in files:
                ext = Path(f).suffix.lower()
                if keyword in f.lower() and ext in EXT_PRIORITY:
                    matches.append(Path(root) / f)
        if matches:
            return sorted(matches, key=lambda p: EXT_PRIORITY.index(p.suffix.lower()))[0]
    raise FileNotFoundError("Global Superstore file not found. On Kaggle, use Add Input to attach the dataset.")


def load_superstore(path):
    """Load the dataset: .txt is tab-separated, .csv comma-separated, .xlsx Excel."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".xlsx":
        return pd.read_excel(path)
    sep = "\t" if ext == ".txt" else ","
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, sep=sep, encoding=enc)
        except UnicodeDecodeError:
            print(f"{enc} failed, trying next encoding...")
    raise ValueError(f"Could not decode {path}")


data_path = find_data_file()
raw = load_superstore(data_path)
print("Loaded:", data_path)
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))

## 2. First look at the data

`head()` and `tail()` to see what rows look like, `info()` for types and non-null counts, and `describe()` for the numeric ranges.

In [ ]:
raw.head()

In [ ]:
raw.tail()

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
# summary of the text columns: how many unique values each one has
raw.describe(include="object").T

### What I noticed

- **Column order is alphabetical** (`Category`, `City`, `Country`, ...), so related fields such as the two dates are far apart. I reorder them during cleaning so the table reads like an order record.
- **Extra helper columns.** There is a column named `记录数` (Chinese for "record count"), plus `Year`, `weeknum` and `Market2`. They look like leftovers from a BI/pivot tool. I need to check whether they add information or just repeat other columns.
- **Dates are strings.** `Order Date` and `Ship Date` load as `object` with values like `2011-01-07 00:00:00.000`. They have to be real datetimes before I can compute shipping time or group by month.
- **`Sales` is an integer.** Sales are rounded to whole dollars, while `Profit` and `Shipping Cost` keep their decimals. `Profit` goes negative, which is expected (loss-making orders).
- **`Customer ID` vs `Customer Name`:** there are far more IDs than names, so the ID is probably not one-per-person.
- **`Sales` and `Profit` are very skewed**: the max is many times the median. I need to decide whether those are outliers to remove or real big orders.

Each check below is followed by the fix, and each fix is a function, so at the end `clean_superstore()` runs the whole cleaning process from the raw file in one call.

## 3. Constant and redundant columns

In [ ]:
# columns with only one value carry no information
nunique = raw.nunique()
constant_cols = list(nunique[nunique == 1].index)
print("Constant columns:", constant_cols)
for c in constant_cols:
    print(f"  {c!r} -> only value: {raw[c].unique()}")

In [ ]:
# Market2 vs Market: does each Market map to exactly one Market2?
print("Market values:", raw["Market"].nunique(), "| Market2 values:", raw["Market2"].nunique())
print("Markets mapping to more than one Market2:",
      (raw.groupby("Market")["Market2"].nunique() > 1).sum())
pd.crosstab(raw["Market"], raw["Market2"])

**Finding:**
- `记录数` has the value `1` on every row. It is a row counter for pivot tables, and `len()` or `count()` gives me the same thing. **Dropped.**
- `Market2` is just `Market` with `US` and `Canada` merged into "North America". Every `Market` maps to exactly one `Market2`, so it adds nothing I can't rebuild with a `map()`. I keep the more detailed `Market` (7 values) and **drop** `Market2`.
- `Year` and `weeknum` should come from `Order Date`. I check that after converting the dates (next step) before dropping them.

In [ ]:
def drop_constant_columns(df):
    """Drop columns that have a single value on every row."""
    nunique = df.nunique()
    const = list(nunique[nunique == 1].index)
    return df.drop(columns=const)


def drop_redundant_columns(df, cols=("Market2", "Year", "weeknum")):
    """Drop helper columns that can be rebuilt from other columns."""
    return df.drop(columns=[c for c in cols if c in df.columns])

## 4. Data types: converting the dates

The date strings look like `2011-01-07 00:00:00.000`. I give `to_datetime` the exact format so nothing is guessed, and use `errors="coerce"` so a bad value turns into `NaT` that I can count instead of breaking the notebook. If an Excel copy is loaded and the dates are already datetimes, they are left alone.

In [ ]:
DATE_COLS = ["Order Date", "Ship Date"]
DATE_FORMAT = "%Y-%m-%d %H:%M:%S.%f"


def convert_dates(df, cols=DATE_COLS, fmt=DATE_FORMAT):
    """Parse the date strings into datetime64 and report anything unparseable."""
    df = df.copy()
    for col in cols:
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col].astype(str).str.strip(), format=fmt, errors="coerce")
        bad = df[col].isna().sum()
        if bad:
            print(f"Warning: {bad} unparseable values in {col}")
    return df


dates = convert_dates(raw[DATE_COLS + ["Year", "weeknum"]])
print(dates.dtypes)
print("NaT values:", dates[DATE_COLS].isna().sum().to_dict())
print("Order Date range:", dates["Order Date"].min().date(), "to", dates["Order Date"].max().date())

Now I can check the helper columns `Year` and `weeknum` against `Order Date`. "Week number" can be defined more than one way, so I test two common ones: the spreadsheet `WEEKNUM` style (week 1 contains January 1, weeks start on Sunday) and the ISO week.

In [ ]:
od = dates["Order Date"]
print("Year mismatches vs Order Date year:", (dates["Year"] != od.dt.year).sum())

# spreadsheet-style WEEKNUM: week 1 contains Jan 1, weeks start on Sunday
jan1 = pd.to_datetime(od.dt.year.astype(str) + "-01-01")
weeknum_sheet = (od.dt.dayofyear - 1 + (jan1.dt.dayofweek + 1) % 7) // 7 + 1
weeknum_iso = od.dt.isocalendar().week.astype(int)

print("weeknum range:", dates["weeknum"].min(), "-", dates["weeknum"].max())
print("Match rate, spreadsheet WEEKNUM: {:.1%}".format((dates["weeknum"] == weeknum_sheet).mean()))
print("Match rate, ISO week:            {:.1%}".format((dates["weeknum"] == weeknum_iso).mean()))

**Finding:** `Year` matches the order date's year on every row, so it is fully redundant. The output above shows which week definition `weeknum` follows. Either way it is derived from `Order Date`, and I can recompute any date part I need with the `.dt` accessor (which I do in Week 6). Both are **dropped** to avoid having two sources of truth for the same fact.

I also convert the text columns with only a few repeated values to `category`, and make `Order Priority` an **ordered** category (Low < Medium < High < Critical) so it sorts correctly later.

In [ ]:
CATEGORY_COLS = ["Ship Mode", "Segment", "Market", "Region", "Category", "Sub-Category"]
PRIORITY_ORDER = ["Low", "Medium", "High", "Critical"]


def convert_categories(df):
    """Low-cardinality text columns to category; priority as ordered category."""
    df = df.copy()
    for col in CATEGORY_COLS:
        df[col] = df[col].astype("category")
    df["Order Priority"] = pd.Categorical(df["Order Priority"], categories=PRIORITY_ORDER, ordered=True)
    return df

## 5. Missing values

In [ ]:
missing = raw.isna().sum()
print("Total missing values:", missing.sum())
missing[missing > 0]

**Finding:** the file has **no missing values** in any column, so there is nothing to fill or drop.

I still add a guard to the pipeline: rows missing any *key* field (IDs, dates, sales, profit) get dropped with `dropna(subset=...)`, because those rows cannot be used in any aggregation. On this file it removes 0 rows, but it protects the pipeline if a later version of the data has gaps. Descriptive fields like city or state would be filled with `"Unknown"` instead of dropping the whole row.

Missing values *created* by my own calculations are handled where they come up. For example, profit margin is undefined when `Sales` is 0 (see the range checks below and Week 6).

In [ ]:
KEY_COLS = ["Order ID", "Order Date", "Ship Date", "Customer ID", "Product ID", "Sales", "Profit", "Quantity"]
TEXT_FILL_COLS = ["City", "State", "Country", "Customer Name", "Product Name"]


def handle_missing(df):
    """Drop rows missing key fields; fill descriptive text fields with 'Unknown'."""
    before = len(df)
    df = df.dropna(subset=[c for c in KEY_COLS if c in df.columns])
    print(f"Rows dropped for missing key fields: {before - len(df)}")
    fill = {c: "Unknown" for c in TEXT_FILL_COLS if c in df.columns}
    return df.fillna(fill)

## 6. Duplicates

First I check for exact duplicate rows. `Row ID` is a unique counter, so I check both with it and without it (two rows that are identical apart from `Row ID` would still be duplicates). Then I check a business key: the same product, in the same order, shipped the same day, in the same quantity.

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicate Row IDs:", raw["Row ID"].duplicated().sum())
print("Duplicates ignoring Row ID:", raw.drop(columns="Row ID").duplicated().sum())

DUP_KEY = ["Order ID", "Product ID", "Ship Date", "Quantity"]
print("Duplicates by", DUP_KEY, ":", raw.duplicated(subset=DUP_KEY).sum())

In [ ]:
# look at the key-based duplicates side by side: which columns differ?
key_dupes = raw[raw.duplicated(subset=DUP_KEY, keep=False)].sort_values(DUP_KEY + ["Row ID"])
differs = key_dupes.groupby(DUP_KEY).nunique().gt(1).sum()
print("Columns that differ inside a duplicate group:\n", differs[differs > 0])
key_dupes[["Row ID", "Order ID", "Product ID", "Quantity", "Sales", "Profit", "Shipping Cost"]].head(6)

**Finding:** there are **no exact duplicates**, not even after ignoring `Row ID`. 9 rows repeat another row on order, product, ship date and quantity. The output above shows that inside those pairs the `Row ID` and `Shipping Cost` differ.

**Decision:** I keep `drop_duplicates()` (ignoring `Row ID`) in the pipeline, so exact repeats in a future version of the file would be removed. I **keep** the 9 near-duplicates. Their shipping costs differ, which looks like a real split shipment rather than a row copied by mistake, and 9 rows out of 51,290 (0.02%) cannot move any aggregate in a meaningful way.

In [ ]:
def remove_duplicates(df):
    """Drop exact duplicates, ignoring the Row ID counter."""
    before = len(df)
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "Row ID"])
    print(f"Duplicates removed: {before - len(df)}")
    return df

## 7. Text columns: whitespace, casing and IDs

### Whitespace and casing

Stray spaces or different casing would make one value look like two different groups in a `groupby`. So I check every text column.

In [ ]:
text_cols = raw.select_dtypes(include="object").columns.drop(DATE_COLS)
ws = {c: int((raw[c] != raw[c].str.strip()).sum()) for c in text_cols}
print("Values with leading/trailing whitespace:", sum(ws.values()))

for c in ["Ship Mode", "Segment", "Market", "Region", "Category", "Sub-Category", "Order Priority"]:
    if raw[c].nunique() != raw[c].str.lower().nunique():
        print("Case conflict in", c)
print("Ship Mode values:", sorted(raw["Ship Mode"].unique()))
print("Market values:", sorted(raw["Market"].unique()))
print("Order Priority values:", sorted(raw["Order Priority"].unique()))

**Finding:** no whitespace problems and no casing conflicts, and the category labels are consistent (`US`, `EU`, `APAC`, ...). I keep a `normalize_strings()` step in the pipeline anyway: it strips spaces and collapses double spaces. On this file it changes nothing, but it is cheap and protects the group-bys.

In [ ]:
def normalize_strings(df):
    """Strip whitespace and collapse repeated spaces in every text column."""
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    return df

### Customer ID vs Customer Name

In [ ]:
print("Unique Customer IDs:  ", raw["Customer ID"].nunique())
print("Unique Customer Names:", raw["Customer Name"].nunique())
print("IDs linked to more than one name:", (raw.groupby("Customer ID")["Customer Name"].nunique() > 1).sum())

ids_per_name = raw.groupby("Customer Name")["Customer ID"].nunique()
print("\nIDs per customer name:")
print(ids_per_name.describe()[["mean", "min", "max"]])

# the IDs for one customer and the markets they appear in
example = ids_per_name.idxmax()
raw.loc[raw["Customer Name"] == example].groupby(["Customer ID", "Market"]).size().rename("lines").head(10)

**Finding:** there are 4,873 customer IDs but only 795 names. No ID belongs to more than one name, but one name has several IDs. The IDs end in an extra digit that changes with the market (for example `LS-172304`), so a `Customer ID` really means "this customer in this market".

**Decision:** I keep both columns as they are. For **customer-level** analysis (like lifetime value in Week 6) I group by `Customer Name`, because grouping by ID would split one person into several "customers". Limitation: if two different real people shared a name they would be merged. I accept that because splitting one person into several market-level "customers" would distort lifetime value a lot more.

### Product ID vs Product Name

In [ ]:
names_per_pid = raw.groupby("Product ID")["Product Name"].nunique()
multi = names_per_pid[names_per_pid > 1]
print("Product IDs:", raw["Product ID"].nunique(), "| Product names:", raw["Product Name"].nunique())
print("Product IDs with more than one name:", len(multi))

# example: one ID and the different names it is used for
pid = multi.index[0]
raw.loc[raw["Product ID"] == pid, ["Product ID", "Product Name", "Sub-Category"]].drop_duplicates()

**Finding:** 457 product IDs are used for more than one product name. In the example above, a single ID covers different items. This is a quirk of how the source data was built, and I can't tell which name is "right".

**Decision:** I do not change the data. I avoid grouping by `Product ID` and use `Product Name` (or `Sub-Category`) as the product label in the analysis.

## 8. Discount values

Retail discounts normally come in round steps, so I list every distinct discount value to look for anything odd.

In [ ]:
disc_counts = raw["Discount"].value_counts().sort_index()
print("Distinct discount values:", len(disc_counts))
# show 3 decimals so values like 0.002 are not hidden by the 2-decimal display setting
print(disc_counts.rename(index="{:.3f}".format).to_string())

In [ ]:
# values ending in ...02 (0.002, 0.202, 0.402, 0.602): where do they come from?
odd = raw["Discount"].round(3).isin([0.002, 0.202, 0.402, 0.602])
print("Rows with an x.x02 discount:", odd.sum())
raw.loc[odd].groupby(["Market", "Discount"]).size().unstack(fill_value=0)

**Finding:** there are 27 distinct discounts between 0 and 0.85. Most are normal steps (0.1, 0.2, 0.5 ...), and some markets use steps like 0.07 / 0.17 / 0.27. A few values look off by a tiny amount: **0.002, 0.202, 0.402, 0.602**. The output above shows which markets they appear in. They read like 0%, 20%, 40% and 60% with a small rounding or tax artifact added.

**Decision:** I keep the raw values because I can't prove what the "true" discount was, and the difference is 0.2 percentage points. When I put discounts into bands in Week 6, I choose bin edges so that 0.202 falls in the same band as 0.20, 0.402 with 0.40, and so on. That way the artifact does not push a row into a higher discount band.

## 9. Validating ranges and checking outliers

Before deciding on outliers I check that the values are at least *possible*: no negative sales, quantity at least 1, discount between 0 and 1, shipping cost not negative, and ship date not before order date.

In [ ]:
checks = {
    "Sales < 0": (raw["Sales"] < 0).sum(),
    "Sales == 0": (raw["Sales"] == 0).sum(),
    "Profit < 0": (raw["Profit"] < 0).sum(),
    "Profit == 0": (raw["Profit"] == 0).sum(),
    "Quantity < 1": (raw["Quantity"] < 1).sum(),
    "Discount outside 0-1": (~raw["Discount"].between(0, 1)).sum(),
    "Shipping Cost < 0": (raw["Shipping Cost"] < 0).sum(),
    "Ship before order": (dates["Ship Date"] < dates["Order Date"]).sum(),
}
ship_days = (dates["Ship Date"] - dates["Order Date"]).dt.days
print({k: int(v) for k, v in checks.items()})
print("Days between order and ship: min", ship_days.min(), "| max", ship_days.max(), "| mean", round(ship_days.mean(), 2))

In [ ]:
# the row(s) with zero sales
raw.loc[raw["Sales"] == 0, ["Order ID", "Product Name", "Quantity", "Discount", "Sales", "Profit"]]

**Finding:** nothing is impossible. There are no negative sales or quantities, every discount is between 0 and 0.85, and every ship date falls 0 to 7 days after its order date. Negative profit (12,544 rows) is a real business outcome, not an error. It is what I want to study.

There is **one row with `Sales` = 0**. `Sales` is rounded to whole dollars, so this is a very cheap item that rounded down to $0. I keep the row because its quantity and profit are still real, but profit margin (`Profit / Sales`) is undefined for it. In Week 6 its margin is set to `NaN` instead of dividing by zero. The rounding also means per-line margins on very small sales are approximate, so I compute group margins from **totals** (sum of profit / sum of sales).

In [ ]:
def iqr_outliers(s, k=1.5):
    """Boolean mask of values outside the IQR fences."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)


num_cols = ["Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]
outlier_summary = pd.DataFrame({
    "iqr_outliers": [iqr_outliers(raw[c]).sum() for c in num_cols],
    "pct_of_rows": [iqr_outliers(raw[c]).mean() * 100 for c in num_cols],
    "median": [raw[c].median() for c in num_cols],
    "max": [raw[c].max() for c in num_cols],
}, index=num_cols)
outlier_summary

In [ ]:
# what do the biggest sales look like?
raw.nlargest(5, "Sales")[["Order ID", "Country", "Sub-Category", "Product Name", "Quantity", "Discount", "Sales", "Profit"]]

**Finding:** the IQR rule flags a lot of rows: 5,655 for `Sales`, 9,755 for `Profit` (about 19%), 5,909 for `Shipping Cost`, 4,172 for `Discount` and 877 for `Quantity`. That is just the shape of retail data: most line items are small and a few are big.

**Decision: keep the outliers.** The largest sales are things like copiers, phones, machines and furniture bought in quantity. They are real orders, not typos (all range checks passed). They are also the lines behind the biggest profits *and* the biggest losses, which is exactly what my question is about, so removing them would hide the most important part of the story. For later charts I will use medians or log scales where the skew gets in the way.

## 10. Running the cleaning pipeline

All the cleaning steps chained into one function, so the whole process can be rerun from the raw file in one call. The last step puts the columns in a logical order (IDs, dates, shipping, customer, location, product, numbers).

In [ ]:
COLUMN_ORDER = ["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Order Priority",
                "Customer ID", "Customer Name", "Segment", "City", "State", "Country", "Market", "Region",
                "Product ID", "Category", "Sub-Category", "Product Name",
                "Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]


def reorder_columns(df, order=COLUMN_ORDER):
    """Put known columns in a logical order; anything else goes at the end."""
    first = [c for c in order if c in df.columns]
    return df[first + [c for c in df.columns if c not in first]]


def clean_superstore(df):
    """Full Week 5 cleaning pipeline: raw dataframe in, clean dataframe out."""
    df = drop_constant_columns(df)
    df = drop_redundant_columns(df)
    df = normalize_strings(df)
    df = handle_missing(df)
    df = remove_duplicates(df)
    df = convert_dates(df)
    df = convert_categories(df)
    df = reorder_columns(df)
    return df.reset_index(drop=True)


df = clean_superstore(raw)
df.info()

In [ ]:
# quick validation of the cleaned data
assert df["Order Date"].notna().all() and df["Ship Date"].notna().all()
assert df.isna().sum().sum() == 0
assert not {"Market2", "Year", "weeknum"} & set(df.columns)
print("Columns: {} -> {}".format(raw.shape[1], df.shape[1]))
print("Date range:", df["Order Date"].min().date(), "to", df["Order Date"].max().date())
print("Rows:", len(df), "| Orders:", df["Order ID"].nunique(),
      "| Customer IDs:", df["Customer ID"].nunique(), "| Customer names:", df["Customer Name"].nunique())
df.head()

### Week 5 cleaning summary

| Issue | What I found | What I did |
|---|---|---|
| File format | Tab-separated `.txt`, columns in alphabetical order | Loader uses `sep="\t"` for `.txt`; columns reordered logically |
| Constant column | `记录数` ("record count") is 1 on every row | Dropped (generic constant-column check) |
| Redundant columns | `Market2` = `Market` with US + Canada merged; `Year` and `weeknum` derived from `Order Date` | Validated against `Market` / `Order Date`, then dropped |
| Date types | Dates stored as `YYYY-MM-DD 00:00:00.000` strings | `pd.to_datetime` with an explicit format, verified 0 `NaT` |
| Categories | Repeated text labels; priority has a natural order | Converted to `category`; `Order Priority` is an ordered category |
| Missing values | None in the file | Guard in pipeline: `dropna` on key fields, `fillna("Unknown")` on descriptive text |
| Duplicates | 0 exact duplicates; 9 key-based near-duplicates differing in shipping cost | `drop_duplicates()` in the pipeline; near-duplicates kept as split shipments |
| Whitespace / casing | None found | `normalize_strings()` kept as a safety step |
| Customer IDs | 4,873 IDs vs 795 names (one ID per customer per market) | Kept; customer-level analysis groups by `Customer Name` |
| Product IDs | 457 IDs used for more than one product name | Kept; analysis uses `Product Name` / `Sub-Category`, not `Product ID` |
| Discount values | Artifacts like 0.002 / 0.202 / 0.402 / 0.602 | Kept raw; Week 6 bins built so they group with 0 / 0.2 / 0.4 / 0.6 |
| Zero sales | 1 row with `Sales` = 0 (rounded to whole dollars) | Kept; its margin will be `NaN`, and group margins come from totals |
| Outliers | Heavy right skew in `Sales` / `Profit` | Range-validated; kept as real large orders |

---
# Week 6: Data Wrangling, Aggregation and Feature Engineering

The data is clean, so now I start shaping it around my question: what drives profit?

## 1. Feature engineering

New columns that the raw data does not have but my analysis needs:

| Feature | How | Why |
|---|---|---|
| `Profit Margin` | `Profit / Sales` (arithmetic, `NaN` when Sales is 0) | Compares profitability across orders of very different sizes |
| `Days to Ship` | `Ship Date - Order Date` in days | Supply chain view: how fast each ship mode really is |
| `Discount Band` | `pd.cut()` into none / low / medium / high | Lets me group by discount level instead of 27 raw values |
| `Order Year`, `Order Month` | `.dt.year`, `.dt.month` | Trends over time and seasonality (replaces the dropped `Year` / `weeknum`) |
| `Is Loss` | `apply()` on `Profit` | Quick flag to count and filter loss-making lines |
| `Ship Speed` | `map()` on `Ship Mode` | Groups the 4 ship modes into expedited vs standard |

The discount bin edges sit at 0, 0.21 and 0.41 rather than exactly 0.2 / 0.4, so the 0.202 and 0.402 artifacts found in Week 5 land in the same band as 0.20 and 0.40.

In [ ]:
DISCOUNT_BINS = [-0.001, 0, 0.21, 0.41, 1.0]
DISCOUNT_LABELS = ["None (0%)", "Low (up to 20%)", "Medium (21-40%)", "High (over 40%)"]
SHIP_SPEED = {"Same Day": "Expedited", "First Class": "Expedited",
              "Second Class": "Standard", "Standard Class": "Standard"}


def add_features(df):
    """Add the engineered columns used in the analysis."""
    df = df.copy()
    # arithmetic: margin per line item, NaN where Sales is 0 (no divide by zero)
    df["Profit Margin"] = (df["Profit"] / df["Sales"].where(df["Sales"] != 0)).astype(float)
    # date math: shipping lead time in days
    df["Days to Ship"] = (df["Ship Date"] - df["Order Date"]).dt.days
    # binning: discount levels
    df["Discount Band"] = pd.cut(df["Discount"], bins=DISCOUNT_BINS, labels=DISCOUNT_LABELS)
    # date parts
    df["Order Year"] = df["Order Date"].dt.year
    df["Order Month"] = df["Order Date"].dt.month
    # apply(): loss flag
    df["Is Loss"] = df["Profit"].apply(lambda p: p < 0)
    # map(): expedited vs standard shipping
    df["Ship Speed"] = df["Ship Mode"].astype(str).map(SHIP_SPEED)
    return df


df = add_features(df)
df[["Order Date", "Ship Date", "Days to Ship", "Sales", "Profit", "Profit Margin",
    "Discount", "Discount Band", "Is Loss", "Ship Mode", "Ship Speed"]].head(8)

In [ ]:
# sanity checks on the new columns
print("Discount band counts:")
print(df["Discount Band"].value_counts().sort_index())
print("\nUnbinned discounts:", df["Discount Band"].isna().sum())
print("Unmapped ship modes:", df["Ship Speed"].isna().sum())
print("Undefined margins (Sales == 0):", df["Profit Margin"].isna().sum())
print("Days to Ship range:", df["Days to Ship"].min(), "-", df["Days to Ship"].max())
print("Share of loss-making lines: {:.1%}".format(df["Is Loss"].mean()))

# which raw discounts ended up in each band?
df.groupby("Discount Band", observed=True)["Discount"].unique()

## 2. Wrangling: selecting, filtering and string operations

### Column subsets

The cleaned table has 30 columns now. For the profit analysis I only need a subset, so I make a narrower working view.

In [ ]:
analysis_cols = ["Order ID", "Order Date", "Order Year", "Market", "Region", "Country",
                 "Segment", "Category", "Sub-Category", "Product Name", "Ship Mode", "Days to Ship",
                 "Sales", "Quantity", "Discount", "Discount Band", "Profit", "Profit Margin",
                 "Shipping Cost", "Is Loss"]
sales = df[analysis_cols]
print(sales.shape)

# iloc slicing: first 5 rows, first 6 columns
sales.iloc[:5, :6]

### Filtering with boolean masks and `loc`

In [ ]:
# loss-making lines that had a high discount
high_disc_losses = sales.loc[(sales["Discount"] > 0.41) & (sales["Is Loss"]),
                             ["Order ID", "Country", "Sub-Category", "Discount", "Sales", "Profit"]]
print(f"High-discount loss lines: {len(high_disc_losses):,} "
      f"({len(high_disc_losses) / sales['Is Loss'].sum():.1%} of all loss lines)")
high_disc_losses.sort_values("Profit").head()

In [ ]:
# how do loss lines compare to profitable ones?
loss_mask = sales["Is Loss"]
print("Average discount on loss lines:      {:.1%}".format(sales.loc[loss_mask, "Discount"].mean()))
print("Average discount on profitable lines: {:.1%}".format(sales.loc[~loss_mask, "Discount"].mean()))

# Technology orders in the US shipped Same Day
us_tech_same_day = sales.loc[(sales["Market"] == "US") & (sales["Category"] == "Technology")
                             & (sales["Ship Mode"] == "Same Day")]
print("US Technology Same Day lines:", len(us_tech_same_day))

# isin() filter on a list of markets
emerging = sales[sales["Market"].isin(["Africa", "EMEA", "LATAM"])]
print("Lines from Africa/EMEA/LATAM:", len(emerging))

### String operations

String operations on the text columns to pull out extra information:
- `Order ID` looks like `CA-2012-124891`: a store/country prefix, the year, and a number. I split it apart and check the year part against `Order Date`.
- The `Customer ID` ends with a market digit (Week 5). Stripping the last character gives a base ID, and comparing its count with the number of names shows whether the market digit is the only thing that splits a person into several IDs.
- Searching `Product Name` for brands, for example how Apple and Samsung products do.

In [ ]:
# split Order ID into its parts: e.g. "CA-2012-124891" -> prefix "CA", year "2012"
id_parts = df["Order ID"].str.split("-", expand=True)
df["Order Prefix"] = id_parts[0].str.upper()
df["Order ID Year"] = pd.to_numeric(id_parts[1], errors="coerce")

print("Order ID year matches Order Date year: {:.2%}".format((df["Order ID Year"] == df["Order Year"]).mean()))
df["Order Prefix"].value_counts().head(8)

In [ ]:
# base customer ID without the trailing market digit
df["Customer Base ID"] = df["Customer ID"].str[:-1]
print("Customer IDs:", df["Customer ID"].nunique(),
      "| base IDs:", df["Customer Base ID"].nunique(),
      "| names:", df["Customer Name"].nunique())

In [ ]:
# brand search in product names (case-insensitive)
brands = ["Apple", "Samsung", "Nokia", "Motorola", "Cisco"]
brand_rows = []
for b in brands:
    m = df["Product Name"].str.contains(b, case=False, regex=False)
    brand_rows.append({"Brand": b, "Lines": m.sum(),
                       "Sales": df.loc[m, "Sales"].sum(), "Profit": df.loc[m, "Profit"].sum()})
brand_summary = pd.DataFrame(brand_rows).set_index("Brand")
brand_summary["Margin"] = brand_summary["Profit"] / brand_summary["Sales"]
brand_summary.sort_values("Sales", ascending=False)

In [ ]:
# products whose name mentions "table": the sub-category that turns out to lose money
tables = df[df["Product Name"].str.contains(r"\btable\b", case=False, regex=True)]
print("Lines with 'table' in the name:", len(tables))
print("Total profit on those lines: {:,.0f}".format(tables["Profit"].sum()))

## 3. Aggregations

### 3a. Profitability by Category and Sub-Category

`groupby` + `agg` with several functions at once: line count, distinct orders, total sales, total profit, average discount, and the share of lines sold at a loss. Margin is computed from the totals (sum of profit / sum of sales). Averaging per-line margins would give tiny orders too much weight, and with `Sales` rounded to whole dollars those small-line margins are not reliable anyway.

In [ ]:
def profit_summary(data, by):
    """Group by `by` and return sales/profit/discount metrics, sorted by profit."""
    out = data.groupby(by, observed=True).agg(
        lines=("Order ID", "count"),
        orders=("Order ID", "nunique"),
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum"),
        avg_discount=("Discount", "mean"),
        loss_rate=("Is Loss", "mean"),
    )
    out["margin"] = out["total_profit"] / out["total_sales"]
    return out.sort_values("total_profit", ascending=False)


by_category = profit_summary(df, "Category")
by_category

In [ ]:
by_subcat = profit_summary(df, ["Category", "Sub-Category"])
by_subcat

### 3b. The discount effect

Grouping by the `Discount Band` feature shows how margin and the share of loss-making lines change as discounts get bigger. The median per-line margin skips the one `NaN` margin automatically.

In [ ]:
by_discount = profit_summary(df, "Discount Band").sort_index()
by_discount["median_line_margin"] = df.groupby("Discount Band", observed=True)["Profit Margin"].median()
by_discount

### 3c. Markets and regions

In [ ]:
by_market = profit_summary(df, "Market")
by_market

In [ ]:
# bottom 5 regions by profit
profit_summary(df, "Region").tail(5)[["total_sales", "total_profit", "avg_discount", "margin"]]

In [ ]:
# countries: top 10 and bottom 10 by total profit
by_country = df.groupby("Country")["Profit"].sum().sort_values()
print("Bottom 10 countries by profit:")
print(by_country.head(10).round(0))
print("\nTop 10 countries by profit:")
print(by_country.tail(10).sort_values(ascending=False).round(0))

### 3d. Shipping: mode, speed and cost

Supply chain view: how long each ship mode actually takes, what it costs relative to the sale, and how it lines up with order priority.

In [ ]:
by_ship = df.groupby("Ship Mode", observed=True).agg(
    lines=("Order ID", "count"),
    avg_days=("Days to Ship", "mean"),
    min_days=("Days to Ship", "min"),
    max_days=("Days to Ship", "max"),
    avg_ship_cost=("Shipping Cost", "mean"),
    total_ship_cost=("Shipping Cost", "sum"),
    total_sales=("Sales", "sum"),
)
by_ship["ship_cost_pct_sales"] = by_ship["total_ship_cost"] / by_ship["total_sales"]
by_ship.sort_values("avg_days")

In [ ]:
# order priority vs ship mode (share of lines in each priority level)
pd.crosstab(df["Order Priority"], df["Ship Mode"], normalize="index").round(3)

### 3e. Trend by year and top customers

In [ ]:
by_year = df.groupby("Order Year").agg(
    orders=("Order ID", "nunique"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
)
by_year["margin"] = by_year["total_profit"] / by_year["total_sales"]
by_year["sales_growth"] = by_year["total_sales"].pct_change()
by_year

For customer lifetime value I group by **`Customer Name`**, not `Customer ID`. As found in Week 5, one person has a separate ID in each market, so grouping by ID would split their history into pieces. `ids` shows how many market-level IDs were combined for each person.

In [ ]:
# customer lifetime value: total profit per customer across all orders and markets
clv = (df.groupby("Customer Name")
         .agg(ids=("Customer ID", "nunique"), markets=("Market", "nunique"), orders=("Order ID", "nunique"),
              total_sales=("Sales", "sum"), lifetime_profit=("Profit", "sum"))
         .sort_values("lifetime_profit", ascending=False))
print("Customers with negative lifetime profit: {} of {}".format((clv["lifetime_profit"] < 0).sum(), len(clv)))
clv.head(10)

### 3f. Key numbers

The headline figures from the aggregations above, collected in one place so the summary below can point to them.

In [ ]:
total_sales, total_profit = df["Sales"].sum(), df["Profit"].sum()
key = {
    "Total sales": f"{total_sales:,.0f}",
    "Total profit": f"{total_profit:,.2f}",
    "Overall margin": f"{total_profit / total_sales:.1%}",
    "Loss-making lines": f"{df['Is Loss'].mean():.1%}",
    "Profit lost in Medium + High discount bands":
        f"{by_discount.loc[['Medium (21-40%)', 'High (over 40%)'], 'total_profit'].sum():,.0f}",
    "Loss rate by discount band": by_discount["loss_rate"].map("{:.0%}".format).to_dict(),
    "Margin by discount band": by_discount["margin"].map("{:.0%}".format).to_dict(),
    "Margin by category": by_category["margin"].map("{:.1%}".format).to_dict(),
    "Loss-making sub-categories": list(by_subcat[by_subcat["total_profit"] < 0].index.get_level_values(1)),
    "Lowest-margin market": by_market["margin"].idxmin(),
    "Highest-margin market": by_market["margin"].idxmax(),
    "Biggest loss countries": list(by_country.head(2).index),
    "Avg days to ship": by_ship["avg_days"].round(1).to_dict(),
    "Shipping cost as % of sales": by_ship["ship_cost_pct_sales"].map("{:.0%}".format).to_dict(),
}
for k, v in key.items():
    print(f"{k}: {v}")

## Week 6 Summary

### What I did

- **Engineered features:** `Profit Margin` (arithmetic, safe for the one zero-sales row), `Days to Ship` (date math), `Discount Band` (`pd.cut` into 4 bins, with edges chosen around the discount artifacts), `Order Year` / `Order Month` (`.dt`), `Is Loss` (`apply`) and `Ship Speed` (`map`).
- **Wrangling:** made a narrower `sales` view with only the columns I need, used `iloc` slicing, filtered with boolean masks, `loc` and `isin()`, and used string operations (`str.split` on `Order ID` for the prefix and year, `.str[:-1]` to get a base customer ID, `str.contains` to search product names).
- **Aggregations:** a reusable `profit_summary()` function that runs `groupby().agg()` with count, nunique, sum and mean, then computes margin from the totals. I ran it by category, sub-category, discount band, market and region. I also grouped by country, ship mode (mean/min/max days and shipping cost), year, and customer name (lifetime profit), plus a priority vs ship mode `crosstab`.

### Why

My question is about **profit**, not just sales. Raw totals favor big categories and big markets, so I mostly compare **margin** (total profit / total sales) and **loss rate** (share of lines sold at a loss). Those show efficiency instead of size. The discount bands and days-to-ship columns turn raw numbers into groups I can compare and later plot. Customer value is grouped by name because of the per-market ID issue from Week 5.

### What the aggregations showed

Overall the store made **$1,467,457 profit on $12,642,905 sales, about an 11.6% margin**, but the profit is very unevenly spread (exact figures are in the key-numbers output above):

1. **Discounts are the biggest profit killer.** Undiscounted lines have the best margin and almost never lose money. Margin drops in the low band, and from the medium band up most lines are sold at a loss. The high band is deeply negative. Together the medium and high bands *lose* money overall, and loss-making lines carry a much higher average discount than profitable ones.
2. **Furniture sells well but earns little.** Furniture brings in more sales than Office Supplies ($4.11M vs $3.79M) but only about a **6.9% margin**, roughly half of Technology (14.0%) and Office Supplies (13.7%). **Tables** is the only sub-category that loses money overall, and it also has a high average discount. Copiers and Phones are the top profit makers.
3. **Geography matters.** APAC and the EU bring in the most profit in dollars, but **EMEA has the lowest margin (about 5.4%)** and the highest average discount. **Canada has the best margin (about 26.6%)** and gets no discounts, which fits the discount finding. The country table shows Turkey and Nigeria as the biggest money-losers.
4. **Shipping works as designed.** Average days to ship go up from Same Day to First, Second and Standard Class (0 to 7 days overall). Expedited modes cost a much larger share of sales in shipping than Standard Class. Low-priority orders ship Standard and Critical orders do not, so priority drives the ship mode.
5. **The business is growing.** Sales and order counts go up every year from 2011 to 2014 while margin stays roughly flat. Even so, some customers have a negative lifetime profit, as the count above the customer table shows.

These results set up the Week 7+ visualizations, especially discount band vs margin and the sub-category profit ranking.

---
# Coming in Week 7+: Visualization, Analysis and Conclusions

*This section is a placeholder and will be filled in over the next weeks.*

Planned work:

- **Visualizations (3+):** bar chart of profit by sub-category, margin by discount band, a monthly sales/profit trend line, profit by market, and a box plot of days to ship by ship mode.
- **Analysis:** connect each chart to the questions in the intro (discounts, products, geography, shipping).
- **Conclusions:** at least 3 conclusions backed by charts and numbers, plus the dataset's limitations (it is a teaching dataset; sales are rounded to whole dollars; product IDs are reused across names; customer IDs are per market; there is no cost or inventory data).